# Auto Loan Credit Analysis Engine
**Mauricio Morales** | Credit Analyst Portfolio Project

This notebook demonstrates a full auto loan credit decision framework — the same logic used by lenders to evaluate applications, calculate rates, and generate credit memos.

**Tools:** Python · Pandas · Matplotlib · Seaborn


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from credit_engine import analyze_application, print_credit_memo, get_tier, monthly_payment

sns.set_theme(style="whitegrid", palette="Blues_d")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'
print("✅ Credit engine loaded")

## Section 1: Interest Rate Structure by Credit Tier

Before approving any loan, lenders set the rate. The rate is built from the base rate plus risk premiums based on credit tier, LTV, loan term, and vehicle age.

In [ ]:
from credit_engine import CREDIT_TIERS, BASE_RATE, TERM_PREMIUMS, ltv_premium, vehicle_age_premium

# Build rate table across tiers and terms
tiers = [t['name'] for t in CREDIT_TIERS]
terms = [36, 48, 60, 72]

rate_data = []
for tier in CREDIT_TIERS:
    for term in terms:
        rate = BASE_RATE + tier['base_premium'] + TERM_PREMIUMS[term]
        rate_data.append({'Tier': tier['name'], 'Term': f'{term}mo', 'Rate': round(rate * 100, 2)})

rate_df = pd.DataFrame(rate_data)
pivot = rate_df.pivot(index='Tier', columns='Term', values='Rate')
pivot = pivot.reindex([t['name'] for t in CREDIT_TIERS])

print("Base Rate: 5.5% | LTV/Age premiums additional\n")
display(pivot.style.background_gradient(cmap='RdYlGn_r').format("{:.2f}%"))


In [ ]:
# Visualize rate by tier (60-month term)
rates_60 = [BASE_RATE + t['base_premium'] + TERM_PREMIUMS[60] for t in CREDIT_TIERS]
colors = ['#22c55e','#86efac','#fbbf24','#f97316','#ef4444']

fig, ax = plt.subplots()
bars = ax.bar(tiers, [r*100 for r in rates_60], color=colors, edgecolor='white')
ax.set_title('APR by Credit Tier (60-Month Term, Base LTV)')
ax.set_ylabel('Interest Rate (%)')
ax.set_ylim(0, max(r*100 for r in rates_60) * 1.2)
for bar, rate in zip(bars, rates_60):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f'{rate*100:.2f}%', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()


## Section 2: The Credit Decision Matrix

In [ ]:
# Simulate 500 applicants and run them through the engine
import random
random.seed(42)

applicants = []
for i in range(500):
    score = random.randint(480, 820)
    income = random.randint(28000, 120000)
    existing_debt = random.randint(0, int(income * 0.25 / 12))
    vehicle_value = random.randint(10000, 55000)
    loan_req = round(vehicle_value * random.uniform(0.80, 1.25), 0)
    vehicle_year = random.randint(2017, 2024)
    term = random.choice([48, 60, 72])

    r = analyze_application(
        applicant_name=f"Applicant {i+1}",
        credit_score=score,
        annual_income=income,
        existing_monthly_debt=existing_debt,
        loan_amount_requested=loan_req,
        vehicle_value=vehicle_value,
        vehicle_year=vehicle_year,
        term_months=term
    )
    applicants.append({
        'credit_score': score,
        'income': income,
        'existing_debt': existing_debt,
        'vehicle_value': vehicle_value,
        'loan_requested': loan_req,
        'vehicle_year': vehicle_year,
        'term': term,
        'tier': r.get('credit_tier','N/A'),
        'decision': r['decision'],
        'rate': r.get('rate', None),
        'ltv': r.get('ltv', None),
        'dti': r.get('back_end_dti', None),
        'monthly_payment': r.get('monthly_payment', None),
        'approved_amount': r.get('approved_amount', None),
    })

df = pd.DataFrame(applicants)
print(f"Simulated {len(df)} applicants")
print(df['decision'].value_counts())


In [ ]:
# Decision breakdown
decision_counts = df['decision'].value_counts()
colors_d = {'APPROVE':'#22c55e','CONDITIONAL APPROVE':'#fbbf24','DECLINE':'#ef4444'}
c = [colors_d.get(k,'#94a3b8') for k in decision_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie
axes[0].pie(decision_counts.values, labels=decision_counts.index,
            autopct='%1.1f%%', colors=c, startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('Overall Decision Distribution')

# By tier
tier_order = [t['name'] for t in CREDIT_TIERS]
tier_decision = df.groupby(['tier','decision']).size().unstack(fill_value=0)
tier_decision = tier_decision.reindex(tier_order)
tier_decision.plot(kind='bar', ax=axes[1],
    color=[colors_d.get(c,'#94a3b8') for c in tier_decision.columns],
    edgecolor='white')
axes[1].set_title('Decision Count by Credit Tier')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(title='Decision')

plt.tight_layout()
plt.show()


## Section 3: Risk Factor Analysis — DTI and LTV

In [ ]:
approved = df[df['decision'].isin(['APPROVE','CONDITIONAL APPROVE'])].copy()
declined = df[df['decision'] == 'DECLINE'].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# DTI distribution
axes[0].hist(approved['dti'].dropna()*100, bins=25, alpha=0.7, color='#22c55e', label='Approved')
axes[0].hist(declined['dti'].dropna()*100, bins=25, alpha=0.7, color='#ef4444', label='Declined')
axes[0].set_title('Back-End DTI Distribution: Approved vs Declined')
axes[0].set_xlabel('DTI (%)')
axes[0].set_ylabel('Count')
axes[0].legend()

# LTV distribution
axes[1].hist(approved['ltv'].dropna()*100, bins=25, alpha=0.7, color='#22c55e', label='Approved')
axes[1].hist(declined['ltv'].dropna()*100, bins=25, alpha=0.7, color='#ef4444', label='Declined')
axes[1].set_title('LTV Distribution: Approved vs Declined')
axes[1].set_xlabel('LTV (%)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()


## Section 4: Rate vs Credit Score

In [ ]:
approved_rates = df[df['rate'].notna()].copy()

fig, ax = plt.subplots()
scatter = ax.scatter(approved_rates['credit_score'], approved_rates['rate']*100,
    c=approved_rates['credit_score'], cmap='RdYlGn', alpha=0.5, s=20)
ax.set_title('Interest Rate vs Credit Score')
ax.set_xlabel('Credit Score')
ax.set_ylabel('APR (%)')
plt.colorbar(scatter, label='Credit Score')
plt.tight_layout()
plt.show()
print("Lower score = higher rate. The spread shows individual variation from LTV, term, and vehicle age premiums.")


## Section 5: Sample Credit Memos

Four real scenarios showing Approve, Conditional, Decline, and Hard Decline.

In [ ]:
scenarios = [
    {"applicant_name": "Maria Gonzalez", "credit_score": 745, "annual_income": 72000,
     "existing_monthly_debt": 450, "loan_amount_requested": 28000,
     "vehicle_value": 30000, "vehicle_year": 2023, "term_months": 60},
    {"applicant_name": "James Walker", "credit_score": 645, "annual_income": 48000,
     "existing_monthly_debt": 800, "loan_amount_requested": 32000,
     "vehicle_value": 30000, "vehicle_year": 2021, "term_months": 72},
    {"applicant_name": "Tyler Brooks", "credit_score": 480, "annual_income": 38000,
     "existing_monthly_debt": 600, "loan_amount_requested": 22000,
     "vehicle_value": 20000, "vehicle_year": 2019, "term_months": 60},
    {"applicant_name": "Angela Reyes", "credit_score": 635, "annual_income": 55000,
     "existing_monthly_debt": 350, "loan_amount_requested": 26000,
     "vehicle_value": 21000, "vehicle_year": 2020, "term_months": 60},
]

for s in scenarios:
    result = analyze_application(**s)
    print_credit_memo(result)


## Section 6: Portfolio Summary Statistics

In [ ]:
summary = df.groupby('tier').agg(
    applications=('credit_score','count'),
    approvals=('decision', lambda x: (x.isin(['APPROVE','CONDITIONAL APPROVE'])).sum()),
    avg_rate=('rate', lambda x: round(x.mean()*100, 2)),
    avg_dti=('dti', lambda x: round(x.mean()*100, 2)),
    avg_ltv=('ltv', lambda x: round(x.mean()*100, 2)),
).reindex([t['name'] for t in CREDIT_TIERS])

summary['approval_rate'] = (summary['approvals'] / summary['applications'] * 100).round(1)
display(summary[['applications','approvals','approval_rate','avg_rate','avg_dti','avg_ltv']].rename(columns={
    'applications':'Apps','approvals':'Approved','approval_rate':'Approval %',
    'avg_rate':'Avg APR %','avg_dti':'Avg DTI %','avg_ltv':'Avg LTV %'
}))


## Key Takeaways

| Insight | What It Means |
|---|---|
| Credit score is the gatekeeper | Determines which lenders will even review the file |
| DTI is the income check | Too much existing debt = decline regardless of score |
| LTV controls collateral risk | High LTV = lender loses money on repossession |
| Rate = base + risk layers | Every risk factor adds basis points |
| Counter offers preserve relationships | Better to counter than decline when one factor is off |

*Project by Mauricio Morales — memora5905.github.io*
